# 1 -- REQUIRED LIBRARIES

In [1]:
pip install yfinance ta pandas numpy

Note: you may need to restart the kernel to use updated packages.


# 2 -- IMPORT LIBRARIES

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

from ta.momentum import RSIIndicator
from ta.trend import EMAIndicator, SMAIndicator, MACD
from ta.volatility import AverageTrueRange

# 3 -- LOAD NSE STOCK LIST CSV

### A -- INSPECT CSV COLUMNS

In [4]:
nse_df = pd.read_csv("All_NSE_Listed_Companies.csv")

print("Available columns:")
print(nse_df.columns.tolist())

Available columns:
['SYMBOL', 'NAME OF COMPANY', ' SERIES', ' DATE OF LISTING', ' PAID UP VALUE', ' MARKET LOT', ' ISIN NUMBER', ' FACE VALUE']


### B -- AUTO-DETECT SYMBOL COLUMN

In [6]:
# Normalize column names
nse_df.columns = nse_df.columns.str.strip().str.upper()

# Possible symbol column names in NSE files
possible_cols = ["SYMBOL", "SECURITY ID", "SECURITYID", "TRADING SYMBOL", "TCKRSYMB"]

symbol_col = None
for col in possible_cols:
    if col in nse_df.columns:
        symbol_col = col
        break

if symbol_col is None:
    raise ValueError("❌ No symbol column found in CSV")

print("✅ Using symbol column:", symbol_col)

✅ Using symbol column: SYMBOL


### C -- CREATE YAHOO-FINANCE SYMBOLS

In [9]:
nse_df["YF_SYMBOL"] = nse_df[symbol_col].astype(str).str.strip() + ".NS"

symbols = nse_df["YF_SYMBOL"].unique().tolist()

print("📊 Total NSE Stocks Loaded:", len(symbols))

📊 Total NSE Stocks Loaded: 2221


# 4 -- FETCH STOCK DATA

In [12]:
# import yfinance as yf

def fetch_stock_data(symbol, period):
    try:
        df = yf.download(symbol, period=period, progress=False)
        if df.empty or len(df) < 30:
            return None

        # Force 1D
        for col in ["Open","High","Low","Close","Volume"]:
            df[col] = df[col].squeeze()

        df.dropna(inplace=True)
        return df

    except:
        return None


# 5 -- ADD TECHNICAL INDICATORS

In [15]:
import pandas as pd
import numpy as np

def add_indicators(df):

    df = df.copy()

    # Ensure numeric
    df["Close"] = pd.to_numeric(df["Close"], errors="coerce")

    # ---------- EMA ----------
    df["EMA_10"] = df["Close"].ewm(span=10, adjust=False).mean()
    df["EMA_20"] = df["Close"].ewm(span=20, adjust=False).mean()
    df["EMA_50"] = df["Close"].ewm(span=50, adjust=False).mean()

    # ---------- RSI ----------
    delta = df["Close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss
    df["RSI"] = 100 - (100 / (1 + rs))

    # ---------- MACD ----------
    ema12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema26 = df["Close"].ewm(span=26, adjust=False).mean()

    df["MACD"] = ema12 - ema26
    df["MACD_SIGNAL"] = df["MACD"].ewm(span=9, adjust=False).mean()

    return df

# 6 -- TREND STRENGTH SCORING (KEY FOR ACCURACY)

In [18]:
def calculate_trend_strength(df):
    """
    Returns:
    trend_score (int)
    trend_strong (bool)
    stock_type (str)
    """

    latest = df.iloc[-1]

    # 🔒 Guaranteed scalars
    close = float(latest["Close"])
    rsi = float(latest["RSI"])
    macd = float(latest["MACD"])
    macd_signal = float(latest["MACD_SIGNAL"])
    stock_type = str(latest["STOCK_TYPE"])

    trend_score = 0

    if stock_type == "LARGE":
        ema20 = float(latest["EMA_20"])
        ema50 = float(latest["EMA_50"])

        if close > ema20 > ema50:
            trend_score += 2
        if macd > macd_signal:
            trend_score += 1
        if 45 <= rsi <= 70:
            trend_score += 1

    elif stock_type == "SMALL":
        ema10 = float(latest["EMA_10"])
        ema20 = float(latest["EMA_20"])

        if close > ema10 > ema20:
            trend_score += 2
        if macd > macd_signal:
            trend_score += 1
        if 40 <= rsi <= 65:
            trend_score += 1

    else:
        recent_avg = float(df["Close"].rolling(5).mean().iloc[-1])
        if close > recent_avg:
            trend_score += 1
        if rsi > 45:
            trend_score += 1

    trend_strong = trend_score >= 3
    return trend_score, trend_strong, stock_type

# 7 -- CONFIDENCE SCORE (0–85%)

In [21]:
def calculate_confidence(df, trend_score, trend_strong, stock_type):

    latest = df.iloc[-1]

    rsi = float(latest["RSI"])
    macd = float(latest["MACD"])
    macd_signal = float(latest["MACD_SIGNAL"])
    volatility = float(df["Close"].pct_change().rolling(5).std().iloc[-1])

    confidence = 0

    # Base
    confidence += 45 if trend_strong else 15

    if stock_type == "LARGE":
        if macd > macd_signal:
            confidence += 15
        if 50 <= rsi <= 65:
            confidence += 15
        elif rsi > 70:
            confidence -= 10
        if volatility < 0.02:
            confidence += 10

    elif stock_type == "SMALL":
        if macd > macd_signal:
            confidence += 10
        if 45 <= rsi <= 60:
            confidence += 10
        elif rsi > 65:
            confidence -= 15
        if volatility > 0.03:
            confidence -= 15

    else:
        confidence -= 10
        if volatility > 0.04:
            confidence -= 15

    # Hard safety
    if not trend_strong:
        confidence = min(confidence, 40)

    confidence = max(0, min(confidence, 85))
    return confidence

# 8 -- BUY / SELL / HOLD DECISION

In [24]:
def make_decision(trend_strong, confidence, stock_type, rsi):

    if stock_type == "LARGE":
        min_conf = 60
    elif stock_type == "SMALL":
        min_conf = 65
    else:
        return "NO BUY"

    if not trend_strong:
        return "HOLD"

    if confidence < min_conf:
        return "HOLD"

    if rsi > 72:
        return "NO BUY"

    return "BUY"

# 9 -- TARGETS & STOP LOSS (ATR BASED)

In [27]:
def calculate_targets(df):

    latest = df.iloc[-1]

    close = float(latest["Close"])
    atr = float(latest["ATR"])

    target_1 = round(close + atr, 2)
    target_2 = round(close + 2*atr, 2)
    target_3 = round(close + 3*atr, 2)
    stop_loss = round(close - atr, 2)

    t1_pct = round((target_1 - close)/close * 100, 2)
    t2_pct = round((target_2 - close)/close * 100, 2)
    t3_pct = round((target_3 - close)/close * 100, 2)
    sl_pct = round((close - stop_loss)/close * 100, 2)

    return {
        "Close": close,
        "Target_1": target_1,
        "Target_2": target_2,
        "Target_3": target_3,
        "Stop_Loss": stop_loss,
        "T1_%": t1_pct,
        "T2_%": t2_pct,
        "T3_%": t3_pct,
        "SL_%": sl_pct
    }

# 10 -- SCAN ALL NSE STOCKS

### A -- FAST PRE-FILTER (NO INDICATORS)

In [31]:
import yfinance as yf
import pandas as pd
import time

filtered_symbols = []

BATCH_SIZE = 50

for i in range(0, len(symbols), BATCH_SIZE):

    batch = symbols[i:i+BATCH_SIZE]

    try:
        data = yf.download(
            tickers=batch,
            period="3mo",
            interval="1d",
            group_by="ticker",
            threads=True,
            progress=False
        )

        if data is None or data.empty:
            continue

        for symbol in batch:

            try:
                df = data[symbol]

                if df.empty or len(df) < 20:
                    continue

                close = float(df["Close"].iloc[-1])
                avg_vol = float(df["Volume"].rolling(20).mean().iloc[-1])

                if close > 50 and avg_vol > 300000:
                    filtered_symbols.append(symbol)

            except:
                continue

    except:
        continue

    time.sleep(0.3)  # anti-ban safety

print("✅ After liquidity filter:", len(filtered_symbols))

✅ After liquidity filter: 384


### B -- FULL ANALYSIS ONLY ON FILTERED STOCKS

In [33]:
import yfinance as yf
import pandas as pd
import numpy as np
import time

analysis_results = []

# 🚀 Limit universe for performance (top liquid stocks only)
MAX_STOCKS = min(300, len(filtered_symbols))
symbols_9b = filtered_symbols[:MAX_STOCKS]

BATCH_SIZE = 25

for i in range(0, len(symbols_9b), BATCH_SIZE):

    batch = symbols_9b[i:i+BATCH_SIZE]

    try:
        data = yf.download(
            tickers=batch,
            period="2y",          # ⚡ reduced from 5y
            interval="1d",
            group_by="ticker",
            threads=True,
            progress=False
        )

        if data is None or data.empty:
            continue

        for symbol in batch:
            try:
                df = data[symbol]

                if df.empty or len(df) < 200:
                    continue

                # Flatten columns
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = df.columns.get_level_values(0)

                close = df["Close"].dropna()
                if len(close) < 200:
                    continue

                # -----------------------------
                # PRICE METRICS
                # -----------------------------
                price = float(close.iloc[-1])
                low_52 = float(close.tail(252).min())
                high_52 = float(close.tail(252).max())
                atl = float(close.min())
                ath = float(close.max())

                # -----------------------------
                # MOVING AVERAGES
                # -----------------------------
                sma50 = float(close.rolling(50).mean().iloc[-1])
                sma200 = float(close.rolling(200).mean().iloc[-1])

                # -----------------------------
                # RSI (FAST & SAFE)
                # -----------------------------
                delta = close.diff()
                gain = delta.where(delta > 0, 0).rolling(14).mean()
                loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
                rs = gain.iloc[-1] / (loss.iloc[-1] + 1e-9)
                rsi = float(100 - (100 / (1 + rs)))

                trend = "Bullish" if price > sma50 > sma200 else "Neutral"

                analysis_results.append({
                    "Symbol": symbol,
                    "Price": round(price, 2),
                    "52W Low": round(low_52, 2),
                    "52W High": round(high_52, 2),
                    "ATL": round(atl, 2),
                    "ATH": round(ath, 2),
                    "RSI": round(rsi, 2),
                    "Trend": trend
                })

            except:
                continue

    except:
        continue

    time.sleep(0.2)   # minimal throttle

final_df = pd.DataFrame(analysis_results)

print("📊 Stocks analyzed:", len(final_df))

📊 Stocks analyzed: 281


### C -- FINAL TOP PICKS

In [35]:
df = final_df.copy()

# ================================
# 1️⃣ DISTANCE METRICS (FIRST!)
# ================================
df["Dist_52W_High_%"] = ((df["52W High"] - df["Price"]) / df["52W High"] * 100).round(2)
df["Dist_52W_Low_%"]  = ((df["Price"] - df["52W Low"]) / df["52W Low"] * 100).round(2)
df["Dist_ATH_%"]      = ((df["ATH"] - df["Price"]) / df["ATH"] * 100).round(2)

# Remove extreme bad data
df = df.replace([np.inf, -np.inf], np.nan).dropna()

# ================================
# 2️⃣ BASE CONFIDENCE (NOW SAFE)
# ================================
def compute_base_confidence(row):

    conf = 30  # starting base

    rsi = row["RSI"]

    # Trend bonus
    if row["Trend"] == "Bullish":
        conf += 20

    # RSI sweet zone
    conf += max(0, 20 - abs(55 - rsi)) * 0.8

    # Distance from ATH (risk control)
    conf += min(row["Dist_ATH_%"], 40) * 0.35

    # Distance from 52W Low (strength)
    conf += min(row["Dist_52W_Low_%"], 50) * 0.25

    return round(conf, 2)

df["Base_Confidence"] = df.apply(compute_base_confidence, axis=1)
df["Base_Confidence"] = df["Base_Confidence"].clip(35, 90)

# ================================
# 3️⃣ WEEKLY SHORT-TERM PICKS
# ================================
weekly_short_top5 = (
    df[
        (df["Trend"] == "Bullish") &
        (df["RSI"].between(55, 70)) &
        (df["Dist_52W_High_%"] < 10)
    ]
    .sort_values("Base_Confidence", ascending=False)
    .head(5)
)

# ================================
# 4️⃣ WEEKLY LONG-TERM PICKS
# ================================
weekly_long_top5 = (
    df[
        (df["Trend"] == "Bullish") &
        (df["RSI"].between(40, 55)) &
        (df["Dist_52W_Low_%"] > 20)
    ]
    .sort_values("Base_Confidence", ascending=False)
    .head(5)
)

# ================================
# 5️⃣ ALL-TIME SHORT-TERM PICKS
# ================================
alltime_short_top5 = (
    df[
        (df["RSI"] < 45) &
        (df["Dist_ATH_%"] > 30)
    ]
    .sort_values("Base_Confidence", ascending=False)
    .head(5)
)

# ================================
# 6️⃣ ALL-TIME LONG-TERM PICKS
# ================================
alltime_long_top5 = (
    df[
        (df["Trend"] == "Bullish") &
        (df["Dist_ATH_%"] > 25)
    ]
    .sort_values("Base_Confidence", ascending=False)
    .head(5)
)

# ================================
# SUMMARY
# ================================
print("✅ STEP 9C FIXED & COMPLETED\n")

print("📌 Weekly Short-Term Picks:")
print(weekly_short_top5["Symbol"].tolist())

print("\n📌 Weekly Long-Term Picks:")
print(weekly_long_top5["Symbol"].tolist())

print("\n📌 All-Time Short-Term Picks:")
print(alltime_short_top5["Symbol"].tolist())

print("\n📌 All-Time Long-Term Picks:")
print(alltime_long_top5["Symbol"].tolist())

✅ STEP 9C FIXED & COMPLETED

📌 Weekly Short-Term Picks:
['MUFIN.NS', 'KIRLOSENG.NS', 'BANKINDIA.NS', 'BHEL.NS', 'INDUSTOWER.NS']

📌 Weekly Long-Term Picks:
['BSOFT.NS', 'MOTHERSON.NS', 'RBLBANK.NS', 'JAMNAAUTO.NS', 'M&M.NS']

📌 All-Time Short-Term Picks:
['NATCOPHARM.NS', 'GODREJPROP.NS', 'PCBL.NS', 'KAJARIACER.NS', 'ADANIENT.NS']

📌 All-Time Long-Term Picks:
['BSOFT.NS', 'MUFIN.NS', 'INDUSINDBK.NS', 'GODAVARIB.NS', 'KIOCL.NS']


# 11 -- FINAL AI RECOMMENDATION ENGINE

In [37]:
def generate_trade_plan(df_subset, label):
    
    print(f"\n{'='*70}")
    print(f"📊 {label.upper()} — AI SELECTED PICKS")
    print(f"{'='*70}")

    for rank, (idx, row) in enumerate(df_subset.iterrows(), start=1):

        price = row["Price"]

        # -----------------------
        # ATR ESTIMATION (safe)
        # -----------------------
        atr = price * 0.03 if row["Trend"] == "Bullish" else price * 0.025

        # -----------------------
        # TARGETS & STOP LOSS
        # -----------------------
        t1 = round(price + atr, 2)
        t2 = round(price + 2*atr, 2)
        t3 = round(price + 3*atr, 2)
        sl = round(price - atr, 2)

        t1_pct = round((t1 - price) / price * 100, 2)
        t2_pct = round((t2 - price) / price * 100, 2)
        t3_pct = round((t3 - price) / price * 100, 2)
        sl_pct = round((price - sl) / price * 100, 2)

        # -----------------------
        # CONFIDENCE SCALING
        # -----------------------
        base_conf = row["Base_Confidence"]

        # Rank-based decay (1st more confident than 5th)
        rank_position = df_subset.index.get_loc(idx) + 1
        rank_decay = rank_position * 1.6   # smooth decay

        confidence = round(
        max(52, min(92, base_conf - rank_decay)),
        2
        )

        # -----------------------
        # FINAL RECOMMENDATION
        # -----------------------
        if confidence >= 78 and row["RSI"] < 68:
            action = "BUY ✅"
        elif confidence >= 68:
            action = "HOLD ⚠️"
        else:
            action = "AVOID ❌"


        # -----------------------
        # PRINT OUTPUT
        # -----------------------
        print(f"\n#{rank}️⃣ Stock : {row['Symbol']}")
        print(f"💰 Price : ₹{price}")
        print(f"📈 Action: {action}")
        print(f"🎯 Confidence : {confidence}%")

        print("\n🎯 Targets:")
        print(f"  Target 1 : ₹{t1}  (+{t1_pct}%)")
        print(f"  Target 2 : ₹{t2}  (+{t2_pct}%)")
        print(f"  Target 3 : ₹{t3}  (+{t3_pct}%)")

        print(f"\n🛑 Stop Loss : ₹{sl}  (-{sl_pct}%)")

        print("\n📊 Key Levels:")
        print(f"  52W Low  : ₹{row['52W Low']}")
        print(f"  52W High : ₹{row['52W High']}")
        print(f"  ATL      : ₹{row['ATL']}")
        print(f"  ATH      : ₹{row['ATH']}")

        print("-" * 60)


# ================================
# RUN FOR ALL CATEGORIES
# ================================

generate_trade_plan(weekly_short_top5, "Weekly Short-Term Picks")
generate_trade_plan(weekly_long_top5, "Weekly Long-Term Picks")
generate_trade_plan(alltime_short_top5, "All-Time Short-Term Picks")
generate_trade_plan(alltime_long_top5, "All-Time Long-Term Picks")


📊 WEEKLY SHORT-TERM PICKS — AI SELECTED PICKS

#1️⃣ Stock : MUFIN.NS
💰 Price : ₹114.58
📈 Action: BUY ✅
🎯 Confidence : 82.01%

🎯 Targets:
  Target 1 : ₹118.02  (+3.0%)
  Target 2 : ₹121.45  (+6.0%)
  Target 3 : ₹124.89  (+9.0%)

🛑 Stop Loss : ₹111.14  (-3.0%)

📊 Key Levels:
  52W Low  : ₹64.62
  52W High : ₹122.56
  ATL      : ₹64.62
  ATH      : ₹269.3
------------------------------------------------------------

#2️⃣ Stock : KIRLOSENG.NS
💰 Price : ₹1218.7
📈 Action: HOLD ⚠️
🎯 Confidence : 73.74%

🎯 Targets:
  Target 1 : ₹1255.26  (+3.0%)
  Target 2 : ₹1291.82  (+6.0%)
  Target 3 : ₹1328.38  (+9.0%)

🛑 Stop Loss : ₹1182.14  (-3.0%)

📊 Key Levels:
  52W Low  : ₹573.59
  52W High : ₹1311.9
  ATL      : ₹573.59
  ATH      : ₹1402.84
------------------------------------------------------------

#3️⃣ Stock : BANKINDIA.NS
💰 Price : ₹143.32
📈 Action: HOLD ⚠️
🎯 Confidence : 71.8%

🎯 Targets:
  Target 1 : ₹147.62  (+3.0%)
  Target 2 : ₹151.92  (+6.0%)
  Target 3 : ₹156.22  (+9.0%)

🛑 Stop Loss 